# Phân vùng polyp: chạy toàn bộ thí nghiệm trên Colab

1. Tải thư mục `Project` lên Google Drive (ví dụ `MyDrive/advanced_ml/Project`).
2. Runtime → Change runtime type → **T4 GPU**.
3. Chạy lần lượt các ô. Kết quả (`outputs/`) nằm trên Drive nên không mất khi phiên bị ngắt; chạy lại ô huấn luyện sẽ tự tiếp tục từ `last.pt`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/advanced_ml/Project'   # sửa nếu bạn để ở chỗ khác
DATA_ROOT = '/content/data/polyp'                        # ổ cục bộ của Colab: đọc ảnh nhanh hơn Drive
%cd {PROJECT}
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q timm==1.0.30 albumentations==2.0.8 gdown
!python -m pytest tests -q

## Dữ liệu
Tải bản PraNet vào ổ cục bộ (mỗi phiên tải lại, khoảng 730 MB), tạo split và kiểm tra ảnh trùng.

In [ ]:
!python scripts/download_data.py {DATA_ROOT}
!ls {DATA_ROOT}
!python scripts/prepare_splits.py --root {DATA_ROOT}
!python scripts/check_duplicates.py --root {DATA_ROOT}


## Huấn luyện baseline
Mỗi mô hình khoảng 1 giờ trên T4 (ước tính). Có thể chia ra nhiều phiên: chạy lại ô sẽ resume.

In [ ]:
MODELS = ['unet', 'multiresunet', 'cenet', 'transunet', 'swinunet']
for m in MODELS:
    !python scripts/train.py --config configs/{m}.yaml --opts data.root={DATA_ROOT}
    !python scripts/evaluate.py --run outputs/runs/{m} --save-pred

## Ablation trên baseline tốt nhất (RQ3)
Đặt `BEST` theo bảng kết quả baseline.

In [ ]:
BEST = 'cenet'
for a in ['c3_domain_aug', 'c2_boundary', 'c2_c3']:
    !python scripts/train.py --config configs/{BEST}.yaml configs/ablation/{a}.yaml --opts data.root={DATA_ROOT}
    !python scripts/evaluate.py --run outputs/runs/{BEST}_{a}

## Thống kê, chi phí tính toán và hình cho báo cáo

In [ ]:
REFERENCE = 'cenet'   # CNN tốt nhất, dùng cho cột p-value của Bảng 5
models = ' '.join(MODELS)
!python scripts/stats_tests.py --models {models} --reference {REFERENCE} --ablation {BEST} {BEST}_c3_domain_aug {BEST}_c2_boundary {BEST}_c2_c3
!python scripts/benchmark.py --configs {' '.join(f'configs/{m}.yaml' for m in MODELS)}
!python scripts/plot_dataset_examples.py --root {DATA_ROOT}
!python scripts/plot_report_figures.py --models {models} --reference {REFERENCE} --root {DATA_ROOT}
!ls outputs/tables report/figures

## Demo

In [ ]:
!pip install -q gradio
!python app/gradio_app.py --run outputs/runs/{BEST} --share